In [1]:
import pandas as pd
from sklearn.svm import SVC
import pandas as pd

In [2]:
df_train = pd.read_csv(r"D:\Study\KLTN\G8Vitamin\data\raw\raw_not_null_train.csv")
df_train_null = pd.read_csv(r"D:\Study\KLTN\G8Vitamin\data\raw\null_train.csv")

df_test = pd.read_csv(r"D:\Study\KLTN\G8Vitamin\data\raw\raw_not_null_test.csv")
df_test_null = pd.read_csv(r"D:\Study\KLTN\G8Vitamin\data\raw\null_test.csv")

In [3]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4644 entries, 0 to 4643
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   SEQN                      4644 non-null   float64
 1   Gender                    4644 non-null   float64
 2   Age                       4644 non-null   float64
 3   Race                      4644 non-null   float64
 4   PIR                       4644 non-null   float64
 5   Weight                    4644 non-null   float64
 6   Height                    4644 non-null   float64
 7   BMI                       4644 non-null   float64
 8   WaistCircumference        4644 non-null   float64
 9   Hba1c                     4644 non-null   float64
 10  FastingGlucose            4644 non-null   float64
 11  Albumin                   4644 non-null   float64
 12  ALT                       4644 non-null   float64
 13  AST                       4644 non-null   float64
 14  Alkaline

In [4]:
removed_coluimns = ['VitaminD', 'YearStart', 'SEQN']
unuseful_columns = [
    "BMI", "ALT",
    "FastingGlucose", "UricAcid", "Hemoglobin",
    "PlateletCount", "MeanPlateletVolume", "TotalCholesterol",
    "MeanCellVolumn", "MeanCellHemoglobin", 
    "RedCellDistributionWidth", "Creatinine"
]

In [5]:
df_train.drop(columns=removed_coluimns + unuseful_columns, inplace=True)
df_train_null.drop(columns=removed_coluimns + unuseful_columns, inplace=True)

df_test.drop(columns=removed_coluimns + unuseful_columns, inplace=True)
df_test_null.drop(columns=removed_coluimns + unuseful_columns, inplace=True)

In [6]:
df_train_null.dropna(subset=['label'], inplace=True)
df_test_null.dropna(subset=['label'], inplace=True)

df_train_null = df_train_null.reset_index(drop=True)
df_test_null = df_test_null.reset_index(drop=True)

In [7]:
print(df_train['label'].value_counts())
print(df_test['label'].value_counts())
print(df_train_null['label'].value_counts())
print(df_test_null['label'].value_counts())

label
0.0    12034
1.0     6635
Name: count, dtype: int64
label
0.0    3203
1.0    1441
Name: count, dtype: int64
label
0.0    13911
1.0     7967
Name: count, dtype: int64
label
0.0    3972
1.0    1872
Name: count, dtype: int64


In [8]:
df_train.columns

Index(['Gender', 'Age', 'Race', 'PIR', 'Weight', 'Height',
       'WaistCircumference', 'Hba1c', 'Albumin', 'AST', 'AlkalinePhosphotase',
       'BUN', 'GGT', 'TotalBilirubin', 'HDLCholesterol', 'Triglycerides',
       'LDLCholesterol', 'Hematocrit', 'milk_consumption', 'label'],
      dtype='object')

## Traning Model

In [9]:
category_columns = [
    'Gender','Race' ,'milk_consumption', 'label'
]

In [10]:
print(len(df_train.columns))

20


In [11]:
# model stacking
from catboost import CatBoostClassifier 
from sklearn.ensemble import GradientBoostingClassifier 
from sklearn.ensemble import RandomForestClassifier 
from sklearn.tree import DecisionTreeClassifier 
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import StackingClassifier 
from lightgbm import LGBMClassifier 
from xgboost import XGBClassifier 
from sklearn.naive_bayes import GaussianNB

dt_setups = { 
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42), 
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42), 
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}

gbc_setups = { 
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42), 
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42), 
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42), 
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42), 
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42), 
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42) 
}

rf_setups = { 
    "Balanced": RandomForestClassifier( n_estimators=100, max_depth=None, random_state=42 ), 
    "ShallowFast": RandomForestClassifier( n_estimators=50, max_depth=5, max_features="sqrt", random_state=42 ), 
    "Deep": RandomForestClassifier( n_estimators=300, max_depth=20, min_samples_split=5, min_samples_leaf=2, random_state=42 ), 
    "Regularized": RandomForestClassifier( n_estimators=100, max_depth=10, min_samples_split=10, min_samples_leaf=4, max_features=0.6, random_state=42 ), 
    "Conservative": RandomForestClassifier( n_estimators=500, max_depth=None, min_samples_split=20, min_samples_leaf=10, max_features="log2", random_state=42 ),
    "RobustSubsample": RandomForestClassifier(
        n_estimators=150, max_depth=15, min_samples_split=5, min_samples_leaf=3, bootstrap=True, max_features="sqrt", random_state=42 
    ), 
    "Lightweight": RandomForestClassifier( n_estimators=50, max_depth=8, max_features=0.5, random_state=42 ), 
    "Heavy": RandomForestClassifier( n_estimators=1000, max_depth=None, min_samples_split=2, min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1 ) 
}

base_learners = [ 
    ('lightgbm', LGBMClassifier(n_estimators=100, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)), 
    ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)), 
    ('RandomForest', rf_setups['Regularized']), 
    ('catboost', CatBoostClassifier( depth=6, learning_rate=0.02, iterations=1000, loss_function='Logloss', verbose=False, eval_metric='Recall', random_state=42 )), 
    # ('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)), 
    # ('SVM', SVC(kernel='rbf', C=10, gamma='auto', class_weight='balanced', probability=True, random_state=42)), 
]


# ====== 3) Meta-learner ====== 
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',          # saga supports L1, L2, elasticnet
    C=0.2,
    random_state=42
    # penalty='elasticnet',   # mix between L1 and L2                 
    # class_weight='balanced',
)


# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  
    n_jobs=1
)

In [12]:
nullable_support = {
    "LightGBM": True,
    "XGBoost": True,
    "GradientBoosting": False,
    "RandomForest": False,
    "Naive Bayes": False,
    "LogisticRegression": False,
    "ElasticNetLR": False,
    "SVM": False,
    "Stacking": False,  
}

classifiers = {
    'LightGBM': LGBMClassifier(
        n_estimators=120, max_depth=6, num_leaves=31, 
        learning_rate=0.05, random_state=42
    ),

    'XGBoost': XGBClassifier(
        n_estimators=80, learning_rate=0.066, 
        random_state=42, verbosity=0
    ),

    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=80, learning_rate=0.05, random_state=42
    ),

    'RandomForest': RandomForestClassifier(
        n_estimators=100, random_state=42
    ),

    'Naive Bayes': GaussianNB(var_smoothing=1e-10),

    'LogisticRegression': LogisticRegression(
        solver='lbfgs', max_iter=500, random_state=42
    ),
    
    'ElasticNetLR': LogisticRegression(
        penalty='elasticnet', solver='saga', l1_ratio=0.5, 
        C=1.0, max_iter=1000, random_state=42
    ),

    'SVM': SVC(kernel='rbf', C=10, gamma=0.01, probability=True, random_state=42),
    
    'Stacking': stacking_clf,
}


In [13]:
import time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, precision_recall_curve, auc,matthews_corrcoef,balanced_accuracy_score,cohen_kappa_score
)

from imblearn.pipeline import Pipeline as ImbPipeline
import os

# ====== 0) Train/test split ======

# for not null data
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

# for null data
X_train_raw_null = df_train_null.drop(columns='label')
y_train_null = df_train_null['label']
X_test_raw_null = df_test_null.drop(columns=['label'])
y_test_null = df_test_null['label']

# numberic columns
numeric_cols = [col for col in df_train.columns if col not in category_columns]

experiment_cases = {
    "noPreprocess": [
        ('classifier', classifiers)
    ]
}

# ====== 1) Wrapper for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)
    
# ====== 2) Train & evaluate ======
rows_detailed = []
rows_summary = []
rows_proba = [] # to save predicted probabilities

data_versions = [
    "Raw_data (100%)",
]


ROOT_DIR = r"D:\Study\KLTN\G8Vitamin\src\models\experiment\results"  
FOLDERNAME = "raw_experiment"
STORE_PATH = os.path.join(ROOT_DIR, FOLDERNAME)
os.makedirs(STORE_PATH, exist_ok=True)

print("RESULTS WILL BE STORED IN:", STORE_PATH)

for version in data_versions:
    for case_name, steps in experiment_cases.items():
        summary_row = {
            "Data versioning": f"{version} - {case_name}",
            "Completeness": "✔",
            "Consistency": "✔",
        }

        for name, clf in classifiers.items():
            print(f"\nTraining {name} on {version} [{case_name}]...")

            # Insert classifier into pipeline steps
            steps_with_clf = [(step, obj) if step != 'classifier' else ('classifier', clf) 
                              for step, obj in steps]

            pipeline = ImbPipeline(steps=steps_with_clf)
            wrapped_model = ImblearnWrapper(pipeline)

            # Get model and check for nullable support
            # supports_null = nullable_support.get(name, False)
            # X_train_raw_use = X_train_raw_null if supports_null else X_train_raw
            # y_train_raw_use = y_train_null if supports_null else y_train
            # X_test_raw_use = X_test_raw_null if supports_null else X_test_raw
            # y_test_raw_use = y_test_null if supports_null else y_test
           
            X_train_raw_use =  X_train_raw
            y_train_raw_use = y_train
            X_test_raw_use = X_test_raw
            y_test_raw_use = y_test

            try:
                # Training
                start = time.time()
                wrapped_model.fit(X_train_raw_use, y_train_raw_use)
                train_time = time.time() - start

                # Testing
                start_test = time.time()
                y_pred = wrapped_model.predict(X_test_raw_use)
                y_proba = wrapped_model.predict_proba(X_test_raw_use)
                test_time = time.time() - start_test

                # Save predicted probabilities
                for idx, (yt, yp, prob) in enumerate(zip(y_test_raw_use.values, y_pred, y_proba)):
                    rows_proba.append({
                        "Data versioning": f"{version} - {case_name}",
                        "Model": name,
                        "Sample_index": idx,
                        "y_true": int(yt),
                        "y_pred": int(yp),
                        "Proba_class0": round(prob[0], 6),
                        "Proba_class1": round(prob[1], 6),
                    })

                # Global metrics
                acc = accuracy_score(y_test_raw_use, y_pred)
                f1_macro = f1_score(y_test_raw_use, y_pred, average='macro', zero_division=0)
                f1_weighted = f1_score(y_test_raw_use, y_pred, average='weighted', zero_division=0)
                acc_balanced = balanced_accuracy_score(y_test_raw_use, y_pred)
                mcc = matthews_corrcoef(y_test_raw_use, y_pred)
                cohen_kappa = cohen_kappa_score(y_test_raw_use, y_pred)

                if len(np.unique(y_test_raw_use)) == 2:
                    auc_roc = roc_auc_score(y_test_raw_use, y_proba[:, 1])
                    auc_pr = auc(*precision_recall_curve(y_test_raw_use, y_proba[:, 1])[1::-1])
                else:
                    auc_roc = roc_auc_score(y_test_raw_use, y_proba, multi_class='ovr', average='macro')
                    auc_pr = np.nan

                # Per-class metrics
                cm = confusion_matrix(y_test_raw_use, y_pred, labels=[0, 1])
                for i, cls in enumerate([0, 1]):
                    TP = cm[i, i]
                    FN = cm[i, :].sum() - TP
                    FP = cm[:, i].sum() - TP
                    TN = cm.sum() - (TP + FN + FP)

                    PPV = TP/(TP+FP) if (TP+FP) > 0 else 0
                    NPV = TN/(TN+FN) if (TN+FN) > 0 else 0
                    SEN = TP/(TP+FN) if (TP+FN) > 0 else 0
                    SPE = TN/(TN+FP) if (TN+FP) > 0 else 0

                    rows_detailed.append({
                        "Data versioning": f"{version} - {case_name}",
                        "Model": name,
                        "Label": cls,
                        "Training time": round(train_time, 4) if cls == 0 else None,
                        "Test time": round(test_time, 4) if cls == 0 else None,
                        "ACC": round(acc, 4) if cls == 0 else None,
                        "ACC balanced": round(acc_balanced, 4) if cls == 0 else None,
                        "MCC": round(mcc, 4) if cls == 0 else None,
                        "CohenKappa": round(cohen_kappa, 4) if cls == 0 else None,
                        "F1_macro": round(f1_macro, 4) if cls == 0 else None,
                        "F1_weighted": round(f1_weighted, 4) if cls == 0 else None,
                        "ROC_AUC": round(auc_roc, 4) if cls == 0 else None,
                        "PR_AUC": round(auc_pr, 4) if cls == 0 else None,
                        "TP": TP, "FP": FP, "FN": FN, "TN": TN,
                        "PPV": round(PPV, 4), "NPV": round(NPV, 4),
                        "SEN": round(SEN, 4), "SPE": round(SPE, 4),
                    })

                    # Summary F1 per label
                    summary_row[f"F1_label{cls}_{name}"] = round(f1_score(y_test_raw_use, y_pred, pos_label=cls), 3)

                summary_row["Training time(seconds)"] = round(train_time, 3)
                summary_row[name] = round(acc, 3)

                print(f"{name} [{case_name}] - ACC={acc:.4f}, F1_macro={f1_macro:.4f}, ROC_AUC={auc_roc:.4f}, PR_AUC={auc_pr:.4f}")

            except Exception as e:
                print(f"ERROR TRAINING {name} [{case_name}]: {e}")

        rows_summary.append(summary_row)

# ====== 3) Save detailed results only ======
excel_measures_path = os.path.join(STORE_PATH, "training_raw_filter_columns.csv")
excel_proba_path =  os.path.join(STORE_PATH, "predicted_probabilities_filter_columns.csv")

df_measures = pd.DataFrame(rows_detailed)
df_measures.to_csv(excel_measures_path, index=False)

df_proba = pd.DataFrame(rows_proba)
df_proba.to_csv(excel_proba_path, index=False)

print(f"\n EXPORTED DETAILED METRICS TO {excel_measures_path}")


RESULTS WILL BE STORED IN: D:\Study\KLTN\G8Vitamin\src\models\experiment\results\raw_experiment

Training LightGBM on Raw_data (100%) [noPreprocess]...
[LightGBM] [Info] Number of positive: 6635, number of negative: 12034
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000403 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2799
[LightGBM] [Info] Number of data points in the train set: 18669, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.355402 -> initscore=-0.595377
[LightGBM] [Info] Start training from score -0.595377
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression [noPreprocess] - ACC=0.6904, F1_macro=0.6071, ROC_AUC=0.6773, PR_AUC=0.4815

Training ElasticNetLR on Raw_data (100%) [noPreprocess]...


c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


ElasticNetLR [noPreprocess] - ACC=0.6970, F1_macro=0.5996, ROC_AUC=0.6794, PR_AUC=0.4853

Training SVM on Raw_data (100%) [noPreprocess]...
SVM [noPreprocess] - ACC=0.6443, F1_macro=0.5458, ROC_AUC=0.5952, PR_AUC=0.3810

Training Stacking on Raw_data (100%) [noPreprocess]...
[LightGBM] [Info] Number of positive: 6635, number of negative: 12034
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000847 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2799
[LightGBM] [Info] Number of data points in the train set: 18669, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.355402 -> initscore=-0.595377
[LightGBM] [Info] Start training from score -0.595377
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Light